# Sesión 1 — Práctica: Prompt Engineering

En esta práctica vas a escribir y mejorar prompts para analizar reviews de productos electrónicos usando la API de Gemini.

**Objetivo**: Experimentar de primera mano cómo la formulación del prompt cambia radicalmente la calidad del output.

**Ejercicios**:
1. Clasificación básica de una review
2. Extracción de información estructurada
3. Mejora del prompt con rol, formato y restricciones
4. Análisis comparativo de múltiples reviews

> **Nota**: Las celdas marcadas con `# TODO` requieren que escribas código. Las demás son código de soporte listo para ejecutar.

## Setup

In [ ]:
%pip install -q google-genai pandas

In [ ]:
from google import genai
import pandas as pd

from google.colab import userdata
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

client = genai.Client(api_key=GOOGLE_API_KEY)

MODEL = 'gemini-2.5-flash'

print('✓ API configurada')

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/ber2/isdi-gen-ai/refs/heads/main/sesion1/amazon_electronics_50.csv'

try:
    sample = pd.read_csv(DATA_URL)
    print('✓ Dataset cargado desde GitHub')
except Exception:
    sample = pd.read_csv('sesion1/data/amazon_electronics_50.csv')
    print('✓ Dataset cargado desde fichero local')

print(f'Muestra lista: {len(sample)} reviews')
print(sample[['rating', 'title', 'text']].head(3))

## Ejercicio 1: Clasificación Básica

Escribe un prompt que clasifique una review como **positiva**, **negativa** o **mixta**.

**Requisito mínimo**: El modelo debe devolver solo la etiqueta, sin explicación.

In [ ]:
review = sample.iloc[0]
print(f'Rating real: {review["rating"]} ★')
print(f'Texto:\n{review["text"]}')

In [ ]:
# TODO: Escribe un prompt para clasificar la review como positiva, negativa o mixta.
# El modelo debe responder SOLO con una de las tres etiquetas.

prompt = """
# TODO: escribe aquí tu prompt
"""

respuesta = client.models.generate_content(model=MODEL, contents=prompt)
print('Clasificación:', respuesta.text)

**Reflexión**: ¿El modelo siguió el formato que pediste? Si no, ¿qué cambiarías en el prompt para forzarlo?

## Ejercicio 2: Extracción de Información Estructurada

Ahora queremos extraer información más rica. Escribe un prompt que devuelva un **JSON** con:
- `clasificacion`: positiva / negativa / mixta
- `producto_mencionado`: qué producto es (inferir del texto)
- `aspectos_positivos`: lista de aspectos que el cliente valora
- `aspectos_negativos`: lista de quejas o puntos de mejora

In [ ]:
# Usamos una review diferente para este ejercicio
review2 = sample.iloc[5]
print(f'Rating real: {review2["rating"]} ★')
print(f'Texto:\n{review2["text"]}')

In [ ]:
# TODO: Escribe un prompt que extraiga la información estructurada indicada arriba.
# Asegúrate de que el modelo responda SOLO con JSON válido.

prompt2 = """
# TODO: escribe aquí tu prompt
"""

respuesta2 = client.models.generate_content(model=MODEL, contents=prompt2)
print(respuesta2.text)

**Bonus**: Intenta parsear el JSON con `json.loads()`. Si falla, ¿qué parte del prompt hace que el modelo incluya texto extra?

In [ ]:
import json

# TODO (opcional): parsear el JSON de la respuesta anterior
# datos = json.loads(respuesta2.text)
# print(datos)

## Ejercicio 3: Mejora del Prompt

El poder del prompt engineering está en los detalles. Escribe **dos versiones** del mismo prompt — una básica y una mejorada — y compara los resultados.

**Tarea**: Pedir al modelo que redacte una respuesta de atención al cliente para la siguiente review con 1 estrella.

**Técnicas a aplicar en la versión mejorada**:
- Asignar un **rol** al modelo
- Especificar el **tono** deseado
- Limitar la **extensión** de la respuesta
- Dar instrucciones sobre qué **no** hacer

In [ ]:
# Buscamos una review negativa (1-2 estrellas)
reviews_negativas = sample[sample['rating'] <= 2]
review_negativa = reviews_negativas.iloc[0] if len(reviews_negativas) > 0 else sample.iloc[10]

print(f'Rating: {review_negativa["rating"]} ★')
print(f'Texto:\n{review_negativa["text"]}')

In [ ]:
# TODO: Versión básica del prompt
prompt_basico = """
# TODO: prompt simple sin mucho detalle
"""

respuesta_basica = client.models.generate_content(model=MODEL, contents=prompt_basico)
print('=== VERSIÓN BÁSICA ===')
print(respuesta_basica.text)

In [ ]:
# TODO: Versión mejorada — con rol, tono, extensión y restricciones
prompt_mejorado = """
# TODO: prompt mejorado con rol, tono, límite de extensión y restricciones
"""

respuesta_mejorada = client.models.generate_content(model=MODEL, contents=prompt_mejorado)
print('=== VERSIÓN MEJORADA ===')
print(respuesta_mejorada.text)

**Reflexión**: ¿Qué cambio tuvo más impacto en la calidad de la respuesta? ¿El rol? ¿El límite de extensión? ¿Las restricciones?

## Ejercicio 4: Análisis Comparativo

Coge **3 reviews** del mismo tipo de producto (puedes filtrar por `parent_asin` o simplemente elegir 3 manualmente) y pide al modelo que sintetice la **opinión agregada** de los clientes.

**Output esperado**:
- Resumen de los puntos en común
- ¿Hay contradicciones entre reviews?
- ¿Qué recomendarías a alguien que esté valorando comprar este producto?

In [ ]:
# Seleccionar 3 reviews del dataset
tres_reviews = sample.iloc[[0, 1, 2]]

for i, (_, r) in enumerate(tres_reviews.iterrows(), 1):
    print(f'--- Review {i} (Rating: {r["rating"]}★) ---')
    print(r['text'][:300] + '...' if len(r['text']) > 300 else r['text'])
    print()

In [ ]:
# TODO: Construye un prompt que incluya las 3 reviews y pida el análisis comparativo.
# Pista: usa un f-string para incluir las reviews dinámicamente en el prompt.

textos = [r['text'] for _, r in tres_reviews.iterrows()]

prompt_comparativo = """
# TODO: construye aquí el prompt con las 3 reviews incluidas
# Recuerda pedir: puntos en común, contradicciones y recomendación final
"""

respuesta_comparativa = client.models.generate_content(model=MODEL, contents=prompt_comparativo)
print(respuesta_comparativa.text)

## Bonus: Experimento Libre

Usa las celdas de abajo para explorar libremente. Ideas:
- Probar el mismo prompt en inglés y en español — ¿cambia la calidad?
- Pedir al modelo que actúe como un cliente enfadado
- Ver qué pasa si das instrucciones contradictorias
- Intentar que el modelo alucine (invent data) y después pedirle que lo corrija

In [ ]:
# Espacio libre para experimentar
mi_prompt = """
# Escribe aquí tu experimento
"""

respuesta = client.models.generate_content(model=MODEL, contents=mi_prompt)
print(respuesta.text)